# Proyecto Aprendizaje de Máquina - Clasificación de Textos por Década

## Integrantes

| Nombre                       | Código    | Correo electrónico           |
|------------------------------|-----------|------------------------------|
| Adrian Velasquez             | 202222737 | a.velasquezs@uniandes.edu.co |
| Andres Botero Ruiz           | 202223503 | a.boteror@uniandes.edu.co    |
| Daniel Vargas López          | 202123892 | d.vargasl2@uniandes.edu.co   |
| Juan David Torres Albarracín | 202317608 | jd.torresa1@uniandes.edu.co  |

# Preparación del entorno de trabajo

In [ ]:
# Imports

import os
import re
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [ ]:
# Constants

DATA = "data/"
MODELS = "models/"
PREDICTIONS = "predictions/"

TRAIN = os.path.join(DATA, "train.csv")
EVAL = os.path.join(DATA, "eval.csv")

BEST_MODEL = os.path.join(MODELS, "best_model.joblib")
PREDICTION_FILE = os.path.join(PREDICTIONS, "predictions.csv")

In [ ]:
# Load
train_data = pd.read_csv( TRAIN )
eval_data = pd.read_csv( EVAL )
print(f"Train: {train_data.shape} | Eval: {eval_data.shape}")

# Exploración de datos

In [ ]:
# Quick EDA

print(f"Total: {len(train_data):,} | Décadas: {len(train_data['decade'].unique())} | Duplicados: {train_data['text'].duplicated().sum()}")

# Limpieza y Preparación

Reutilizamos el pipeline de preprocesamiento del Proyecto 1, que demostró ser efectivo para manejar artefactos OCR del texto histórico en español. Adicionalmente:

- **Codificación de etiquetas**: las décadas se mapean a índices enteros (0-38)
- **División estratificada**: 80% entrenamiento / 20% validación, manteniendo la distribución de clases
- **Pesos de clase**: para compensar el desbalance entre décadas

In [ ]:
def fix_ocr_artifacts(text):
    """Colapsa espacios entre caracteres aislados — artefacto común en OCR histórico."""
    return re.sub(r'(?<=[A-Za-záéíóúüñÁÉÍÓÚÜÑ]) (?=[A-Za-záéíóúüñÁÉÍÓÚÜÑ])', '', text)

def normalize_text(text):
    text = fix_ocr_artifacts(text)
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'\b\d{4}\b', ' YEAR ', text)
    text = re.sub(r'\d+', ' NUM ', text)
    text = re.sub(r'([.!?,;:])\1+', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_data['text_clean'] = train_data['text'].apply(normalize_text)
eval_data['text_clean']  = eval_data['text'].apply(normalize_text)

# Limpieza básica
train_data = train_data.drop_duplicates(subset=['text'])
train_data = train_data.dropna(subset=['text_clean'])
train_data = train_data[train_data['text_clean'].str.len() > 10].reset_index(drop=True)

print(f'Datos limpios: {len(train_data):,}')

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_data['decade'])
num_classes = len(label_encoder.classes_)
print(f'Clases: {num_classes} décadas ({label_encoder.classes_[0]} – {label_encoder.classes_[-1]})')

X_texts = train_data['text_clean'].values

X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    X_texts, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {len(X_train_texts):,} | Validación: {len(X_val_texts):,}')

# Pesos de clase para manejar el desbalance
class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights_arr))
print(f'Peso mín: {class_weights_arr.min():.3f} | Peso máx: {class_weights_arr.max():.3f}')

# Modelos de Deep Learning

La hipótesis central es que los textos históricos en español codifican información temporal en múltiples niveles:

- **Nivel de carácter**: ortografía, diacríticos, uso de grafemas obsoletos (e.g., ſ, ligaduras)
- **Nivel de palabra**: vocabulario, morfología, frecuencia de arcaísmos
- **Nivel sintáctico**: estructuras oracionales típicas de cada época

Exploramos tres iteraciones de modelos de deep learning:

| Iteración | Modelo               | Representación     | Justificación |
|-----------|----------------------|--------------------|---------------|
| 1         | MLP sobre TF-IDF     | N-gramas de chars  | Baseline DL equivalente al Proy 1 |
| 2         | CNN a nivel de carácter | Secuencia de chars | Aprende filtros convolucionales análogos a n-gramas |
| 3         | BiLSTM a nivel de palabra | Secuencia de palabras | Captura dependencias de largo alcance |

## Iteración 1: MLP sobre TF-IDF de n-gramas de caracteres

El primer modelo de deep learning usa las mismas características TF-IDF de n-gramas de caracteres del Proyecto 1, pero alimentadas a un perceptrón multicapa (MLP) en lugar de un LinearSVC.

**Decisiones de diseño:**
- `max_features=20 000`: suficiente para capturar los n-gramas más discriminativos sin exceder la memoria RAM como matriz densa
- Tres capas densas con BatchNorm + Dropout: regularización para evitar sobreajuste
- `class_weight`: compensar desbalance entre décadas

Este modelo establece si el MLP puede extraer valor no-lineal de las mismas características que LinearSVC usa linealmente.

In [ ]:
TFIDF_FEATURES = 20_000

print('Extrayendo características TF-IDF...')
tfidf_vec = TfidfVectorizer(
    analyzer='char', ngram_range=(2, 4),
    max_features=TFIDF_FEATURES, sublinear_tf=True, min_df=2
)
X_tfidf_train = tfidf_vec.fit_transform(X_train_texts).toarray().astype(np.float32)
X_tfidf_val   = tfidf_vec.transform(X_val_texts).toarray().astype(np.float32)
print(f'Forma TF-IDF: {X_tfidf_train.shape}')

def build_mlp(input_dim, num_classes):
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(512, activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    return keras.Model(inp, out, name='MLP_TF-IDF')

mlp_model = build_mlp(TFIDF_FEATURES, num_classes)
mlp_model.summary()

In [ ]:
mlp_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_mlp = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

history_mlp = mlp_model.fit(
    X_tfidf_train, y_train,
    validation_data=(X_tfidf_val, y_val),
    epochs=30, batch_size=128,
    callbacks=callbacks_mlp,
    class_weight=class_weight_dict,
    verbose=1
)

y_pred_mlp = np.argmax(mlp_model.predict(X_tfidf_val, verbose=0), axis=1)
acc_mlp = accuracy_score(y_val, y_pred_mlp)
print(f'\nMLP TF-IDF — Accuracy validación: {acc_mlp:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history_mlp.history[metric],     label='Entrenamiento')
    ax.plot(history_mlp.history[f'val_{metric}'], label='Validación')
    ax.set_title(f'MLP TF-IDF — {title}')
    ax.set_xlabel('Época')
    ax.set_ylabel(title)
    ax.legend()
plt.tight_layout()
plt.show()

## Iteración 2: CNN a nivel de carácter

El segundo modelo implementa una **Red Neuronal Convolucional (CNN) sobre secuencias de caracteres**. A diferencia del MLP sobre TF-IDF, este modelo:

1. **Aprende embeddings de caracteres** específicos para la tarea, en lugar de usar conteos TF-IDF predefinidos
2. **Captura patrones locales** mediante filtros convolucionales 1D: cada filtro aprende a detectar un patrón de caracteres discriminativo
3. **Multi-escala**: usamos filtros de tamaños 3, 5, 7, 10 y 15, análogos al `ngram_range=(2,6)` del Proyecto 1
4. **GlobalMaxPooling**: extrae la activación máxima de cada filtro independientemente de la posición en el texto

**Longitud máxima: 1500 caracteres** — cubre el percentil 95 de la distribución observada en el EDA.

In [ ]:
MAX_CHAR_LEN  = 1500
CHAR_EMBED_DIM = 64
NUM_FILTERS    = 256
FILTER_SIZES   = [3, 5, 7, 10, 15]

# Vocabulario de caracteres construido únicamente del conjunto de entrenamiento
all_chars  = set()
for text in train_data['text_clean']:
    all_chars.update(text)

char_vocab  = ['<PAD>', '<UNK>'] + sorted(list(all_chars))
char2idx    = {c: i for i, c in enumerate(char_vocab)}
CHAR_VOCAB_SIZE = len(char_vocab)
print(f'Vocabulario de caracteres: {CHAR_VOCAB_SIZE}')

def encode_chars(texts, char2idx, max_len):
    unk = char2idx['<UNK>']
    seqs = []
    for text in texts:
        seq = [char2idx.get(c, unk) for c in text[:max_len]]
        seq += [0] * (max_len - len(seq))
        seqs.append(seq)
    return np.array(seqs, dtype=np.int32)

print('Codificando textos...')
X_char_train = encode_chars(X_train_texts, char2idx, MAX_CHAR_LEN)
X_char_val   = encode_chars(X_val_texts,   char2idx, MAX_CHAR_LEN)
print(f'Shape: {X_char_train.shape}')

In [ ]:
def build_char_cnn(vocab_size, max_len, num_classes,
                   embed_dim=64, num_filters=256, filter_sizes=(3, 5, 7, 10, 15)):
    """
    CNN multi-escala a nivel de carácter.
    Cada rama convolucional aprende n-gramas de distinto orden;
    GlobalMaxPool extrae la característica más relevante de cada filtro.
    """
    inp = keras.Input(shape=(max_len,), name='char_input')

    emb = layers.Embedding(vocab_size, embed_dim, name='char_emb')(inp)
    emb = layers.SpatialDropout1D(0.1)(emb)

    pools = []
    for fs in filter_sizes:
        c = layers.Conv1D(num_filters, fs, activation='relu', padding='same', name=f'conv_{fs}')(emb)
        p = layers.GlobalMaxPooling1D(name=f'pool_{fs}')(c)
        pools.append(p)

    x = layers.Concatenate(name='concat')(pools)

    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    out = layers.Dense(num_classes, activation='softmax', name='output')(x)
    return keras.Model(inp, out, name='CharCNN')

char_cnn = build_char_cnn(CHAR_VOCAB_SIZE, MAX_CHAR_LEN, num_classes,
                          CHAR_EMBED_DIM, NUM_FILTERS, FILTER_SIZES)
char_cnn.summary()

In [ ]:
char_cnn.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        os.path.join(MODELS, 'char_cnn_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

print('Entrenando Char CNN...')
history_cnn = char_cnn.fit(
    X_char_train, y_train,
    validation_data=(X_char_val, y_val),
    epochs=40, batch_size=64,
    callbacks=callbacks_cnn,
    class_weight=class_weight_dict,
    verbose=1
)

y_pred_cnn = np.argmax(char_cnn.predict(X_char_val, verbose=0), axis=1)
acc_cnn    = accuracy_score(y_val, y_pred_cnn)
print(f'\nChar CNN — Accuracy validación: {acc_cnn:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history_cnn.history[metric],          label='Entrenamiento')
    ax.plot(history_cnn.history[f'val_{metric}'], label='Validación')
    ax.set_title(f'Char CNN — {title}')
    ax.set_xlabel('Época')
    ax.set_ylabel(title)
    ax.legend()
plt.tight_layout()
plt.show()

## Iteración 3: BiLSTM a nivel de palabra

La tercera iteración usa un **BiLSTM (LSTM Bidireccional)** sobre secuencias de palabras. Este modelo complementa al CNN porque:

1. **Dependencias de largo alcance**: las LSTM pueden preservar información a lo largo de muchas palabras
2. **Bidireccionalidad**: el texto se procesa en ambas direcciones, capturando contexto izquierdo y derecho
3. **Vocabulario dinámico**: el modelo aprende embeddings para las palabras más frecuentes del corpus

**Limitación esperada**: el vocabulario histórico varía enormemente entre siglos, lo que puede dificultar la generalización de embeddings de palabras. Por eso, el CNN de caracteres es la apuesta principal.

**Longitud máxima: 150 palabras** — captura el contexto suficiente sin un costo computacional prohibitivo.

In [ ]:
MAX_WORD_LEN   = 150
WORD_VOCAB_SIZE = 50_000
WORD_EMBED_DIM  = 128

print('Tokenizando (nivel de palabra)...')
word_tokenizer = Tokenizer(num_words=WORD_VOCAB_SIZE, oov_token='<OOV>')
word_tokenizer.fit_on_texts(X_train_texts)

X_word_train = pad_sequences(
    word_tokenizer.texts_to_sequences(X_train_texts),
    maxlen=MAX_WORD_LEN, padding='post', truncating='post'
)
X_word_val = pad_sequences(
    word_tokenizer.texts_to_sequences(X_val_texts),
    maxlen=MAX_WORD_LEN, padding='post', truncating='post'
)
print(f'Shape: {X_word_train.shape}')

In [ ]:
def build_bilstm(vocab_size, max_len, num_classes, embed_dim=128):
    """
    BiLSTM de dos capas sobre secuencias de palabras.
    SpatialDropout1D en embeddings y Dropout en capas densas para regularización.
    """
    inp = keras.Input(shape=(max_len,), name='word_input')

    emb = layers.Embedding(vocab_size, embed_dim, mask_zero=True, name='word_emb')(inp)
    emb = layers.SpatialDropout1D(0.2)(emb)

    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.2), name='bilstm_1'
    )(emb)
    x = layers.Bidirectional(
        layers.LSTM(64, dropout=0.2), name='bilstm_2'
    )(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    out = layers.Dense(num_classes, activation='softmax', name='output')(x)
    return keras.Model(inp, out, name='BiLSTM')

bilstm_model = build_bilstm(WORD_VOCAB_SIZE, MAX_WORD_LEN, num_classes, WORD_EMBED_DIM)
bilstm_model.summary()

In [ ]:
bilstm_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_bilstm = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

print('Entrenando BiLSTM...')
history_bilstm = bilstm_model.fit(
    X_word_train, y_train,
    validation_data=(X_word_val, y_val),
    epochs=30, batch_size=64,
    callbacks=callbacks_bilstm,
    class_weight=class_weight_dict,
    verbose=1
)

y_pred_bilstm = np.argmax(bilstm_model.predict(X_word_val, verbose=0), axis=1)
acc_bilstm    = accuracy_score(y_val, y_pred_bilstm)
print(f'\nBiLSTM — Accuracy validación: {acc_bilstm:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history_bilstm.history[metric],          label='Entrenamiento')
    ax.plot(history_bilstm.history[f'val_{metric}'], label='Validación')
    ax.set_title(f'BiLSTM — {title}')
    ax.set_xlabel('Época')
    ax.set_ylabel(title)
    ax.legend()
plt.tight_layout()
plt.show()

## Comparación de modelos

Comparamos las tres iteraciones de deep learning entre sí y contra el baseline del Proyecto 1 (LinearSVC con n-gramas de caracteres, accuracy en validación ≈ 0.2957).

In [ ]:
BASELINE_ACC = 0.2957

resultados = pd.DataFrame({
    'Modelo':        ['LinearSVC + CharNgrams (Proy 1)', 'MLP sobre TF-IDF', 'CNN a nivel de carácter', 'BiLSTM a nivel de palabra'],
    'Tipo':          ['Clásico ML (baseline)',            'Deep Learning',    'Deep Learning',           'Deep Learning'],
    'Acc Validación': [BASELINE_ACC,                      acc_mlp,            acc_cnn,                   acc_bilstm]
})

resultados = resultados.sort_values('Acc Validación', ascending=False).reset_index(drop=True)
print(resultados.to_string(index=False))

# Gráfico de comparación
fig, ax = plt.subplots(figsize=(10, 4))
df_plot = resultados.sort_values('Acc Validación', ascending=True)
colores = ['#e74c3c' if t == 'Clásico ML (baseline)' else '#3498db' for t in df_plot['Tipo']]
bars = ax.barh(df_plot['Modelo'], df_plot['Acc Validación'], color=colores, height=0.5)
ax.axvline(BASELINE_ACC, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Baseline LinearSVC ({BASELINE_ACC})')
for bar, val in zip(bars, df_plot['Acc Validación']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', fontsize=10)
ax.set_xlabel('Accuracy en validación')
ax.set_title('Comparación de modelos — Clasificación por Década')
ax.legend()
plt.tight_layout()
plt.show()

best_row   = resultados.iloc[0]
print(f'\nMejor modelo: {best_row["Modelo"]}  |  Acc: {best_row["Acc Validación"]:.4f}')
mejora = best_row['Acc Validación'] - BASELINE_ACC
print(f'Mejora sobre baseline: {mejora:+.4f} ({mejora / BASELINE_ACC * 100:+.1f}%)')

In [ ]:
# Reporte detallado del mejor modelo DL
model_map = {'MLP sobre TF-IDF': y_pred_mlp, 'CNN a nivel de carácter': y_pred_cnn, 'BiLSTM a nivel de palabra': y_pred_bilstm}

# El mejor puede ser cualquiera de los tres modelos DL
dl_resultados = resultados[resultados['Tipo'] == 'Deep Learning']
best_dl_name  = dl_resultados.iloc[0]['Modelo']
best_dl_preds = model_map[best_dl_name]

print(f'Reporte de clasificación — {best_dl_name}')
print(classification_report(
    y_val, best_dl_preds,
    target_names=[str(c) for c in label_encoder.classes_],
    digits=3
))

# Exportación del modelo y predicciones

Guardamos el mejor modelo en formato Keras (`.keras`) y generamos las predicciones sobre el conjunto de evaluación (`eval.csv`) en el formato requerido por la competencia Kaggle.

In [ ]:
# Selección del mejor modelo según accuracy de validación
scores     = {'mlp': acc_mlp, 'cnn': acc_cnn, 'bilstm': acc_bilstm}
best_key   = max(scores, key=scores.get)
best_acc_v = scores[best_key]
print(f'Modelo seleccionado: {best_key.upper()} (acc val = {best_acc_v:.4f})')

# Preparar entrada del conjunto de evaluación
if best_key == 'cnn':
    X_eval_enc   = encode_chars(eval_data['text_clean'].values, char2idx, MAX_CHAR_LEN)
    best_model   = char_cnn
elif best_key == 'bilstm':
    X_eval_enc   = pad_sequences(
        word_tokenizer.texts_to_sequences(eval_data['text_clean'].values),
        maxlen=MAX_WORD_LEN, padding='post', truncating='post'
    )
    best_model   = bilstm_model
else:  # mlp
    X_eval_enc   = tfidf_vec.transform(eval_data['text_clean'].values).toarray().astype(np.float32)
    best_model   = mlp_model

# Guardar modelo
best_model.save(BEST_MODEL_PATH)
print(f'Modelo guardado: {BEST_MODEL_PATH}')

# Generar y guardar predicciones
print('Generando predicciones sobre eval.csv...')
y_probs       = best_model.predict(X_eval_enc, verbose=0)
y_pred_idx    = np.argmax(y_probs, axis=1)
y_pred_decades = label_encoder.inverse_transform(y_pred_idx)

predictions_df = pd.DataFrame({'id': eval_data['id'], 'answer': y_pred_decades})
predictions_df.to_csv(PREDICTION_FILE, index=False)
print(f'Predicciones guardadas: {PREDICTION_FILE}  ({len(predictions_df)} filas)')
print(f'\nMuestra:')
print(predictions_df.head(10).to_string(index=False))

In [ ]:
# Guardar objetos auxiliares necesarios para inferencia futura
aux = {
    'label_encoder': label_encoder,
    'char2idx':      char2idx,
    'word_tokenizer': word_tokenizer,
    'tfidf_vec':     tfidf_vec,
    'best_model_key': best_key,
    'MAX_CHAR_LEN':  MAX_CHAR_LEN,
    'MAX_WORD_LEN':  MAX_WORD_LEN,
    'TFIDF_FEATURES': TFIDF_FEATURES
}
joblib.dump(aux, os.path.join(MODELS, 'inference_objects.joblib'))
print('Objetos de inferencia guardados:', os.path.join(MODELS, 'inference_objects.joblib'))